In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# PPO in detail

PPO is one of the earliest (and thus, the "default") Rl method for post-training LLMs.

It has a lot of pieces which may make it intimidating.

In this module, we try to explain PPO in detail.

# Components

| Persona | Mathematical Signature | Role in the Loop | Status |
| :--- | :--- | :--- | :--- |
| **Actor** | $\pi_\theta(s_t) \rightarrow P(a_t)$ | **The Performer:** Samples the next token. This is the final model. | **Active** (Updates) |
| **Old Actor** | $\pi_{old}(s_t) \rightarrow P(a_t)$ | **The Ghost:** A frozen snapshot of the Actor for the PPO Ratio. | **Static** (Per Iteration) |
| **Reference** | $\pi_{ref}(s_t) \rightarrow P(a_t)$ | **The Moral Compass:** The original SFT model. Used for KL Tax. | **Frozen** (Permanent) |
| **Critic** | $V_\phi(s_t) \rightarrow \text{Value}$ | **The Accountant:** Predicts "Expected Future Profit" ($V$). | **Active** (Updates) |

where 
- $P(a_t)$ denotes the probability distribution over actions
    - For generative LLMs: action = token

## The Three Policies ($\pi$): Actor, Old, Reference

The **Actor**, **Old**, and **Reference** models all share the same structural signature. 
- and Loss function

When you give them a sequence of tokens ($s_t$), 
they all output a probability distribution over the vocabulary.

* **Actor:** 
    - Role: generate the next action/token
    - Update frequency: every mini-batch

    - "Based on my current training, I want to say *this* token next."
    
* **Old:** 
    - Role: serves as an anchor to prevent the updates from deviating too much in this epoch
    - Update frequency: every epoch
    
    - "This is what I would have said at the beginning of this training batch."
    
* **Reference:** 
    - Role: serves as an anchor to prevent updates from deviating too much from the *original* (SFT trained) model
    - Update frequency: never

    - "This is what a helpful, safe human-trained model would say."

**The Logical Comparison:**

We compare the **Actor** to the **Old** to ensure our updates aren't too explosive (the PPO Clip).

We compare the **Actor** to the **Reference** to ensure we aren't "Reward Hacking" (the KL Penalty).

---

| Persona | Update Frequency | Purpose |
| :--- | :--- | :--- |
| **Actor** | **Every Mini-batch** | Continuous learning. It adjusts its weights slightly after seeing a small slice of data to maximize the Advantage. |
| **Old Actor** | **Every PPO Iteration** | Stability. It is "refreshed" only after a full set of epochs (e.g., after 4 passes through the buffer) to become the new baseline for the next round. |
| **Reference** | **Never** | Consistency. It remains the original SFT model throughout the entire training process to ensure the model doesn't drift into gibberish. |

---

### The Actor vs. Old Actor "Drift"

1. **Step 0:** At the start of a PPO iteration, **Actor == Old Actor**.
2. **Step 1:** We update the **Actor** on the first mini-batch. Now it is slightly different from the **Old Actor**.
3. **Step 2:** We calculate the "Probability Ratio" ($r_t$) between the **Actor** (new) and the **Old Actor** (frozen snapshot).
4. **Step 3:** If the **Actor** drifts too far from the **Old Actor** (e.g., > 20%), the PPO Clip kicks in and stops the update.
5. **End of Iteration:** Once we've finished all epochs, we finally update the **Old Actor** by copying the current **Actor** weights into it. The cycle repeats.

---

### Why this matters for the Critic

The **Critic** usually updates at the same frequency as the **Actor** (every mini-batch). 

However, because the **Old Actor** is frozen for the duration of the iteration, the data in the "Experience Buffer" (the tokens we collected) was generated by that fixed policy. 

This means the **Critic** is essentially training on "Recorded History" while the **Actor** is trying to write the "Future."

## The Critic ($V$)

The **Critic** is the outlier. 

Its signature (and Loss function) is fundamentally different because its goal isn't language — it's **valuation**.

When you give it a sequence of tokens ($s_t$), 
it  outputs a prediction of 
- the return/reward to go (cumulative future rewards from this state).


* **Input:** The same state ($s_t$) the Actor sees.
* **Output:** A single scalar number ($V$).

**The Optimization Difference:**

While the three $\pi$ models are evaluated based on **Log-Probabilities**, 
the Critic is evaluated based on **Mean Squared Error** ($MSE$):

$$L_{critic} = \text{MSE}(V_\phi(s_t), \text{Returns})$$

That is
- minimize error between 



### Return vs. Reward-to-Go

In the context of the Critic and PPO, the terms **Return** and **Reward-to-Go** are used interchangeably. 
They both represent the target the Critic is trying to predict.

#### 1. The Mathematical Definition
The **Return** ($G_t$) at time step $t$ is the sum of all future rewards:
$$G_t = r_t + r_{t+1} + r_{t+2} + \dots + r_T$$

#### 2. Why "To-Go"?
If you are halfway through generating a sentence ($t=10$), the rewards you earned for tokens 1 through 9 are "sunk costs." They cannot be changed. The Critic only cares about the **Reward-to-Go**: the potential value still left on the table from token 10 until the end of the sequence ($T$).

#### 3. The Critic's Objective
The Critic ($V_\phi$) attempts to learn a mapping from the current state to this expected return:
$$V_\phi(s_t) \approx \mathbb{E}[G_t \mid s_t]$$


### Comparison: Reward vs. Return

| Term | Scope | Analogy |
| :--- | :--- | :--- |
| **Reward** ($r_t$) | Immediate: The score for a single action. | A single paycheck. |
| **Return / Reward-to-Go** ($G_t$) | Cumulative: The sum of all future scores. | Your total projected career earnings from today forward. |

## Summary: The "Sync" Challenge

Because the **Actor** and **Critic** are the only two personas that are actively learning, they are the only two that can "fall out of sync." 

1. **The Reference** is a fixed point in the past (The SFT baseline).
2. **The Old Actor** is a fixed snapshot of the current iteration.
3. **The Actor and Critic** are in a constant race. As the Actor changes its strategy, the Critic must "sprint" to update its Value Map to match that new strategy.



# Computing the Advantage : The Journey of the "Surprise" 

To isolate how PPO assigns credit to specific steps/tokens
- we start with these simplifying assumptions:

**Simplifying assumptions**

* **No Discounting ($\gamma = 1$):** A reward received 50 tokens from now is just as valuable as a reward received right now.
* **Terminal Reward Only ($r_{<T} = 0$):** No points are given for intermediate tokens. Only the final token receives a score (e.g., $+1.0$).
* **Trace Decay ($\lambda = 1$):** We define $\lambda$ as the parameter that determines how much we "trust" the full trajectory. At $\lambda=1$, we trust the **Total Episode Reward** completely to define our Advantage.

**Justification:** 

These assumptions allow us to 
- see how a single final score is "backpropagated" to specific intermediate actions 
- without the noise of time-decay or mid-process penalties.


## Local Surprise (TD Error)

The "Local Surprise" $\delta_t$ is mathematically defined as the **Temporal Difference (TD) Error** ($\delta_t$). 

$$\delta_t = V(s_{t+1}) - V(s_t)$$

That is, the local surprise is
- how much the action taken at $s_t$
- changed (for the better or worse) the return

Interpretation of the surprise:

1.  **If $\delta_t > 0$:** The Actor did something **better** than the Critic expected. The Actor is rewarded.
2.  **If $\delta_t < 0$:** The Actor did something **worse** than the Critic expected. The Actor is discouraged.
3.  **If $\delta_t = 0$:** The Actor did exactly what was expected. No "Surprise" means no "Advantage," so no change to the weights.

Recall the Equation for TD Error (Local Surprise) $\delta_t$
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$


The Local Surprise has two components:

* **The Reality:** The immediate reward ($r_t$) plus the Critic's estimate of the *next* state ($V(s_{t+1})$).
* **The Expectation:** The Critic's estimate of the *current* state ($V(s_t)$).


> **Translation:** "The value I see now ($r_t + V_{next}$) minus the value I predicted a moment ago ($V_{current}$)."

### Deeper Dive: Value-Based vs. Policy-Based

If we were doing Value-Based TD Learning
- the local surprise would be used to update our estimate of the value function $V$
- and our policy would be derived from the Value function
    - in state $s_t$: choose the action that leads to successor state $s_{t+1}$ with maximal value
    

#### 1. Pure TD Learning (Value-Based)
* **The Intelligence:** Resides entirely in the **Value Function** ($Q$ or $V$).
* **The Policy:** Is a simple "Slave" to the value function. It just looks at all possible next states and picks the one with the highest number.
    * *Equation:* $a_t = \text{argmax}_{a} Q(s_t, a)$
* **The Problem:** In an LLM, the "action space" is 100,000+ tokens. Calculating the value of every single possible word before picking one is computationally impossible in real-time.

#### 2. Actor-Critic (PPO)
* **The Intelligence:** Resides in the **Actor** ($\pi$). The Actor learns the "Vibe" or "Strategy" of which words are good without having to consult a value table for every single token.
* **The Role of TD Error:** Instead of using the TD Error to tell a value-table which number to change, we use it to tell the **Actor** which *neural weights* to strengthen.


#### The Comparison: Who Holds the Map?

| Feature | Pure TD Learning | Actor-Critic (PPO) |
| :--- | :--- | :--- |
| **Who is Smart?** | The Map (Value Function). | The Traveler (Actor). |
| **The Strategy** | "Follow the highest number." | "I've learned that this path feels right." |
| **Handling Scale** | Terrible (Must check every action). | Excellent (The Actor just "knows"). |
| **The Signal** | TD Error updates the **Map**. | TD Error updates the **Traveler**. |




## Advantage $\hat{A}_t$ = Total surprise

Once the episode ends and we see the real reward ($R_{final}$), 
we calculate the Total Surprise. 

Mathematically, 
- the Total Surprise (Advantage)
- is exactly the sum of every Local Surprise from that point forward:

$$\hat{A}_t = \delta_t + \delta_{t+1} + \dots + \delta_T$$


 This is the "Reality Check." 
 - It’s the difference between what we *thought* would happen at token $t$ 
 - and what actually happened at the finish line.
 
Under our implifying assumptions:
$$\hat{A}_t = R_{final} - V(s_t)$$


# Using the Advantage: The Gradient Update

In the Actor's loss function, 
- we use the **Total Surprise** ($\hat{A}_t$) 

as the multiplier for the gradient:

$$\nabla_\theta L \approx \hat{A}_t \nabla_\theta \log \pi_\theta(a_t | s_t)$$

* **If $\hat{A}_t$ is positive:** We "push" the Actor to make that choice more likely.
* **If $\hat{A}_t$ is negative:** We "pull" the Actor away from that choice.

---

## Numerical Example: "The Brilliant Start, The Tragic End"

Imagine a model writing a 3-token code snippet.
* **Start:** Critic expects a mediocre result: $V(s_1) = 0.5$.
* **Step 1:** Actor writes a brilliant function header. Critic is impressed: $V(s_2) = 0.9$.
* **Step 2:** Actor writes a standard line. Critic stays steady: $V(s_3) = 0.9$.
* **Step 3:** Actor makes a typo that breaks the code. Reward Model gives: $R_{final} = 0.2$.

### Calculating Local Surprises ($\delta$):
1.  **$\delta_1$ (The Genius):** $V(s_2) - V(s_1) = 0.9 - 0.5 = \mathbf{+0.4}$
2.  **$\delta_2$ (The Neutral):** $V(s_3) - V(s_2) = 0.9 - 0.9 = \mathbf{0.0}$
3.  **$\delta_3$ (The Blunder):** $R_{final} - V(s_3) = 0.2 - 0.9 = \mathbf{-0.7}$

Calculating Advantage ($\hat{A}_t$) for the Update:
 
* **$\hat{A}_1$ (For token 1):** $\delta_1 + \delta_2 + \delta_3 = 0.4 + 0.0 + (-0.7) = \mathbf{-0.3}$
* **$\hat{A}_3$ (For token 3):** $\delta_3 = \mathbf{-0.7}$

**The Conclusion:** 

Even though the whole episode was a "failure" ($0.2 < 0.5$), 
- the first token only gets a small penalty ($-0.3$) because the Critic "defended" it with a positive local surprise. 
- The third token takes the massive hit ($-0.7$).

## Essence of PPO Advantage

Even in the extreme case where
- there is a single end-of-episode reward
    - no intermediate rewards
    
the PPO Advantage is able to
- assign credit (decompose the Episode Reward)
- to each step/token

rather than equally distributing it to the steps
- as is done in GRPO

**This is surgical credit assignment.**

### Surgical vs. Holistic Credit Assignment

| Feature | PPO (Surgical) | GRPO (Holistic) |
| :--- | :--- | :--- |
| **The Baseline** | **The Critic's Prediction:** A unique value $V$ for every single token. | **The Group Mean:** A single average score $\mu$ for the entire group of 16-64 completions. |
| **The Advantage** | **Token-Level:** Calculated via the sum of TD Errors ($A_t$). Each word gets its own unique "Surprise" score. | **Sentence-Level:** Calculated as the standardized score ($A_i = \frac{R_i - \mu}{\sigma}$). Every token in a "winning" sentence gets the same reward. |
| **Computation** | **Heavy:** Requires a separate Critic model (often as large as the Actor) running alongside it. | **Light:** No Critic model. You just run multiple Actor completions in parallel. |

### Comparison to  GRPO Philosophy: "A Rising Tide Lifts All Tokens"

In GRPO, if an Actor produces a completion that earns a high reward ($R=1.0$) compared to the group average ($\mu=0.5$), 
- the math tells the model:
> "Everything you said in this specific completion was good. Strengthen the weights for **every single token** in this sequence."

#### The Risk of GRPO (The "Passenger" Problem)

Because GRPO attributes success to the whole sentence, 
- a "bad" token can hitch a ride on a "good" sentence. 

* *Example:* If the model writes a brilliant essay but includes one typo, 
- GRPO will still "reward" the typo because the overall essay score was high.

#### The Strength of GRPO
By removing the Critic, you save **50% of your VRAM**. 

This allows you to:
1. Increase the **Group Size** (e.g., compare 64 versions of the same prompt).
2. Use that extra memory to train a much **Larger Actor**.


### The GRPO Advantage Equation
Notice there are no $V(s_t)$ terms here. For the $i$-th completion in a group:

$$A_i = \frac{R_i - \text{mean}(R_{group})}{\text{std}(R_{group})}$$

Every token $a_{i,t}$ within that completion $i$ will use this same $A_i$ value during the policy gradient update.

##  Removing the Simplifying Assumptions (Practical PPO)

To transition to "Production-Grade" PPO, we re-introduce the following:

### A. Re-introducing the Discount Factor ($\gamma < 1$)

We multiply future surprises by $\gamma \approx 0.99$.
* **Effect:** The Advantage at token $t$ cares *more* about the surprises that happen immediately after it and *less* about what happens 500 tokens later.

### Re-introducing the KL Penalty (Intermediate Rewards)

The KL Penalty is 
- an intermediate negative reward 
- applied at **every single token** during the generation process. 

It measures 
- how much the current **Actor** ($\pi_\theta$)
- has deviated from the original **Reference** model ($\pi_{ref}$).

The penalty $r_{kl, t}$ at token $t$ is defined as the KL Divergence between 
- the Actor's probability distribution 
- and the Reference model's distribution:

$$r_{kl, t} = -\beta \cdot \text{KL}(\pi_\theta(a_t \mid s_t) \| \pi_{ref}(a_t \mid s_t))$$

In simpler terms, for a specific action $a_t$:
$$r_{kl, t} = -\beta \cdot \left( \log \frac{\pi_\theta(a_t \mid s_t)}{\pi_{ref}(a_t \mid s_t)} \right)$$

* **$\beta$ (Beta):** The "Stiffness" of the leash. A high $\beta$ keeps the model very close to the original SFT version.
* **The Log Ratio:** If the Actor makes a word 10x more likely than the Reference model did, the penalty grows.

-

The KL Penalty serves three critical functions that keep RL training from collapsing:

**Preventing Reward Hacking**

Reward Models are "proxy" judges. They aren't perfect. 

If we only reward the final outcome, 
- the model will find "loopholes."
    - It might discover that repeating a certain word 100 times or using bizarre gibberish 
    
    results in a high score from the Reward Model. This is called **Reward Hacking**.

The KL Penalty discourages this by serving as a "tax" for each token produced. 

The cost of "being weird" outweighs the benefit of the fake reward.

**Preserving General Capabilities (Catastrophic Forgetting)**

During RL, we are often optimizing for a specific task (e.g., "Be more polite"). 

Without a KL Penalty, the model might "forget" how to code or do math 
- because those weights aren't being specifically reinforced. 

The Reference model acts as an anchor of previous knowledge.

**Ensuring "Smooth" Training**

In the Actor-Critic framework, 
- if the Actor changes too fast, the Critic can't keep up (the "Sync" problem). 

The KL Penalty ensures that 
- the Actor only explores the space "near" its known safe starting point
-  making the learning process stable.

### How it affects the TD Error ($\delta_t$)

In a world with KL Penalties, the "Surprise" at every token is now a balance of **Behavior** and **Value**:

$$\delta_t = \underbrace{(r_{kl, t} + \gamma V(s_{t+1}))}_{\text{The New Reality}} - \underbrace{V(s_t)}_{\text{The Prediction}}$$

Every token the Actor produces now "costs" a little bit of KL Tax. 

To have a positive Advantage
- the Actor must produce a token 
- that is so good
- it justifies the cost of deviating from the Reference model.

We add a small negative reward $r_{kl}$ (KL Penalty) at **every** token 
- to prevent the model from becoming "too weird."

$$\delta_t = (r_{kl} + \gamma V(s_{t+1})) - V(s_t)$$

### Generalized Advantage Estimation (GAE - $\lambda < 1$)

If the Critic is "noisy," 
- we don't trust the long-term sum perfectly.

We blend the local surprise and the total sum using $\lambda$.
$$\hat{A}_t = \delta_t + (\gamma\lambda)\delta_{t+1} + (\gamma\lambda)^2\delta_{t+2} + \dots$$

This is similar to the "learning rate" used to update weights in Gradient Descent.


## 5. Transitioning to Intermediate Rewards (The KL Penalty)

In real LLM training, we add a **KL Divergence Penalty** at every token to keep the model human-like. This changes our reward from "Terminal Only" to "Intermediate."



---

###  From "Tournament" to "Business": Introducing the KL Penalty

In our initial "Spherical Cow" model, we assumed $r_t = 0$ for all intermediate steps. Why? Because it makes the math of **Credit Assignment** perfectly clean. However, in real-world LLM training, we must introduce a **KL Divergence Penalty**.

### Why omit it initially?
1. **Mathematical Clarity:** Without intermediate rewards, the Advantage is a simple subtraction: $R_{final} - V(s_t)$. This is the best way to see the **Telescoping Sum** in action.
2. **The "Pure" Signal:** It allows us to see how the Critic reacts to a "Genius Move" vs. a "Blunder" based solely on the final outcome.

### Why add it now? (The Problem of Reward Hacking)
If we only reward the final outcome, the model will find "loopholes." It might discover that repeating a certain word 100 times or using bizarre gibberish results in a high score from the Reward Model. This is called **Reward Hacking**.

The **KL Penalty** is a small negative reward ($r_{kl}$) applied to **every single token**. It acts as a "tax" on being different from the original, human-like reference model.



---


### Updated Numerical Example (with -0.1 KL Penalty per step):

The Local Surprise must now account for the "reward in hand":
$$\delta_t = r_t + V(s_{t+1}) - V(s_t)$$

* **$V(s_1)=0.5, V(s_2)=0.9, V(s_3)=0.9$**
* **$r_1=-0.1, r_2=-0.1, r_3=0.1$ (Final 0.2 minus 0.1 KL)**

1. **$\delta_1 = -0.1 + 0.9 - 0.5 = +0.3$** (Brilliance, slightly taxed)
2. **$\delta_2 = -0.1 + 0.9 - 0.9 = -0.1$** (Standard move, costs more than it earns)
3. **$\delta_3 = 0.1 - 0.9 = -0.8$** (The blunder, even more painful)

**The Identity Holds:**
The Total Advantage $\hat{A}_1$ is still the sum of future surprises:
$$\hat{A}_1 = 0.3 + (-0.1) + (-0.8) = -0.6$$
This is exactly the same as: **(Actual Total Rewards) - (Initial Expectation)**
$$( (-0.1) + (-0.1) + 0.1 ) - 0.5 = -0.1 - 0.5 = -0.6$$


# The Divergence Actor and the Critic

At the start of RLHF, the **Actor** and **Critic** are identical twins, both born from the **SFT Reference Model**. However, as training begins, they drift apart to fulfill their specific roles:

1. **The Actor (The Performer):** - **Loss:** Policy Gradient.
   - **Evolution:** It learns the *nuances* of what the Reward Model likes. It becomes more strategic and focused on "winning" the episode.
   
2. **The Critic (The Judge):**
   - **Loss:** Mean Squared Error (MSE).
   - **Evolution:** It becomes a "Value Map." It learns to look at a half-finished sentence and accurately predict if it’s going to end in a $+1.0$ or a $-1.0$.

### Why do they diverge?
If the Critic stayed identical to the Actor, it would just be "rooting" for the Actor rather than "judging" it. By updating independently, the Critic can remain an objective auditor, even when the Actor tries something new or "risky."

# Pseudo-code for PPO

This puts the "Surgical" concepts into the actual context of a training loop, illustrating exactly where the **Actor**, **Critic**, and **Reference** models interact.

---

## The Engine Room: Inner vs. Outer Loops

To understand how PPO actually trains, we must look at the two distinct "gears" of the algorithm: the **Outer Loop** (Experience) and the **Inner Loop** (Optimization).

### The Pseudo-Code Logic

    # --- INITIALIZATION ---
    # Actor (π_θ), Critic (V_φ), Reference (π_ref), Reward (RM)

    # --- THE OUTER LOOP (The Experience Phase) ---
    # Goal: The model "acts" in the world and gathers feedback.
    for iteration in range(max_iterations):

        # 1. ROLLOUT: Actor generates a batch of completions.
        # We save the 'logprobs_old' and 'values_old' from this specific version.
        sequences, logprobs_old, values_old = actor.generate(prompts)

        # 2. EVALUATION: The external and internal judges speak.
        rewards_terminal = reward_model.score(sequences)
        kl_penalties = compute_kl(sequences, actor, ref_model) # Intermediate rewards

        # 3. SURGICAL PREP: Calculate the Advantage (Total Surprise)
        # This uses the telescoping sum of local surprises (δ)
        # δ_t = (r_kl + V_next) - V_now
        advantages = compute_gae(kl_penalties, rewards_terminal, values_old)
        returns = advantages + values_old # The 'Target' for the Critic

        # --- THE INNER LOOP (The Optimization Phase) ---
        # Goal: Use the surgical feedback to refine the brain.
        for epoch in range(ppo_epochs):
            for batch in shuffle(sequences, advantages, returns, logprobs_old):

                # A. ACTOR UPDATE: "The Performance Review"
                # Compare current probs to logprobs_old.
                # If Advantage is positive, push probabilities up (with PPO Clipping).
                policy_loss = actor.compute_loss(batch, advantages, logprobs_old)
                actor.step(policy_loss)

                # B. CRITIC UPDATE: "The Accounting Audit"
                # The Critic tries to minimize the error between its guess (V)
                # and the actual outcome (returns).
                value_loss = critic.compute_mse(batch.values, batch.returns)
                critic.step(value_loss)

        # After the Inner Loop, the Actor is slightly better.
        # It becomes the 'Old' model for the next Outer Loop iteration.

---

# Key Concept Recap for the Notebook

| Phase | Purpose | Concept Applied |
| --- | --- | --- |
| **Outer Loop** | Data Collection | **Reference Model** calculates KL penalties to keep the model human-like. |
| **Advantage Calculation** | Credit Assignment | **Telescoping Sum** turns final rewards and KL taxes into a "Surgical" signal for every token. |
| **Inner Loop** | Weight Update | **Old Model** ensures the Actor doesn't change too much in a single step (PPO Clip). |
| **Critic Update** | Learning the Map | The **Critic** learns to be a better "Accountant" by comparing its guesses to the actual results. |

## Why is this so expensive?

By looking at this loop, you can see the **VRAM footprint**:

1. **Actor:** Being updated (Gradient storage required).
2. **Critic:** Being updated (Gradient storage required).
3. **Reference Model:** Constant forward passes for KL.
4. **Reward Model:** Constant forward passes for scores.



## **The Sync Warning: The Risk of the Lagging Judge**

In a perfect world, the **Actor** and **Critic** would always be in sync. But in PPO, they are constantly drifting apart to fulfill their specific roles. This creates a specific risk you should be aware of: **Policy Lag**.

### **The Problem: The "Outdated Map"**
When the **Actor** takes a gradient step, it becomes a "New Actor" with a new strategy. However, the **Critic** is still using a "Value Map" trained on the **Old Actor's** behavior.

* **The Risk:** If the Actor changes too much, the Critic’s advice becomes a **hallucination**. It starts penalizing moves that are actually good in the new strategy, or rewarding moves that have become obsolete. This is why RL training can suddenly "collapse" into gibberish.



### **The Solution: Why we let them diverge anyway**
If we forced the Critic to stay perfectly in sync (by sharing weights or updating them identically), we would create an **Echo Chamber**.
* If the Actor makes a mistake, the "Sync" Critic would simply agree with it because they share the same underlying logic.
* The error would be amplified until the model loses its "Moral Compass."

**The Divergence is a Feature:** We need the Critic to be an **Independent Auditor**. It needs its own set of weights so it can "watch" the Actor's changes and objectively measure: *"Did that weight change actually result in more reward, or did you just get lucky?"*

### **PPO’s Safety Rails**
To manage this "Sync Gap," PPO uses two specific mechanisms:
1.  **The PPO Clip:** We "leash" the Actor, preventing it from changing its policy by more than (usually) **20%** in a single update. This ensures the Critic’s "Old Map" is still "mostly" accurate.
2.  **Multiple Inner Epochs:** We let the Critic "sprint" during the optimization phase, updating its Value Map several times to catch up to the Actor's new behavior before we go out and collect more data.



# PPO vs. GRPO: Moving from "Surgical" to "Statistical"

As models scale to 70B+ parameters, the VRAM cost of PPO becomes a massive bottleneck. **GRPO (Group Relative Policy Optimization)** was introduced to solve this by fundamentally changing how we calculate the "Advantage."

## The Structural Shift: Firing the Accountant
The biggest difference is the removal of the **Critic** (the Value Head).

* **PPO (The Accountant):** Relies on a separate neural network (the Critic) to guess the "Value" of every single token. This requires extra VRAM and creates the "Twin Divergence" problem.
* **GRPO (The Statistics):** Instead of a Critic, GRPO generates a **Group** of outputs (usually 8 or 16) for the same prompt. It calculates the mean score of that group and defines the "Advantage" based on how much better or worse an individual output is compared to its peers.

##  Comparison Table: The Core Trade-offs

| Feature | PPO (Proximal Policy Optimization) | GRPO (Group Relative Policy Optimization) |
| :--- | :--- | :--- |
| **Advantage Source** | **Learned:** The Critic's $V(s)$ prediction. | **Calculated:** Group Mean/Std Dev of rewards. |
| **Model Count** | 4 (Actor, Critic, Ref, Reward). | **2-3** (Actor, Ref, Reward). |
| **Precision** | **Surgical:** Can assign credit to a specific word. | **Coarse:** Primarily judges the whole sequence. |
| **VRAM Usage** | **Very High:** Value Head + Critic gradients. | **Low:** Reverts to standard SFT structure. |
| **Stability** | High (if Critic is well-trained). | High (uses the group as a natural baseline). |

## The "Advantage" Math: A New Perspective

In PPO, we used the **Local Surprise** ($\delta_t$) to find the Advantage. In GRPO, we use **Relative Performance**:

$$A_i = \frac{r_i - \text{mean}(R)}{\text{std_dev}(R)}$$

In GRPO, 
- the model doesn't ask "Is this token better than I expected?" 
- Instead, it asks: 
    - **"Is this entire completion better than the other 7 versions I just wrote?"**



## Why move to GRPO?

1.  **Memory Savings:** By removing the Critic, you save roughly **25-33%** of your VRAM, allowing you to train larger models on the same hardware.
2.  **No "Tug-of-War":** Since there is no Critic sharing the transformer backbone, the weights only have one goal: to follow the Actor's policy.
3.  **Simplicity:** You no longer have to worry about the "Sync Warning" or the Critic "sprinting" to catch up. The group mean is always a "fresh" and accurate baseline for the current Actor.

##  The Verdict

* **Use PPO** when you have a complex environment where **token-level precision** is critical (like dense code generation or complex logic where one specific word breaks everything).
* **Use GRPO** when you are scaling to **massive models** and need an efficient way to align them with human preferences without the overhead of an "Accountant."


# HuggingFace TRL

Here is  how **TRL** handles the architectural heavy lifting for the Critic model.



## The "Heavy Lifting": How TRL Builds the Critic

In practice, you don't need to manually perform surgery on your model’s neural network layers. Libraries like HuggingFace **TRL** (Transformer Reinforcement Learning) automate the process of turning a standard **SFT (Supervised Fine-Tuning)** model into an **Actor-Critic** pair.

### The `AutoModelForCausalLMWithValueHead` Wrapper
The primary tool TRL uses is a specialized wrapper class. Instead of loading a standard language model, you load the model using this class:

```python
from trl import AutoModelForCausalLMWithValueHead

# This loads the SFT model and automatically attaches a "Value Head"
model = AutoModelForCausalLMWithValueHead.from_pretrained("your-sft-model-path")
```

#### **What the Library Does Automatically:**
* **The Backbone:** It loads the full pre-trained transformer (the "Body").
* **The Actor Head:** It keeps the existing Language Modeling head (the "Voice") that predicts the next token.
* **The Critic Head (The Surgery):** It identifies the final hidden layer of the transformer and attaches a **new, randomly initialized linear layer**.
* **The Output:** This new layer is a "Scalar Head"—it takes the high-dimensional vector from the transformer and reduces it to a single number: the **Value $V(s)$**.



---

###  Shared vs. Separate Backbones
One of the most important "heavy lifting" decisions TRL makes is **Weight Sharing**.

* **Shared "Trunk":** By default, the Actor and the Critic share the same transformer body (e.g., the 7B parameters). They only diverge at the very end with their specific "heads."
* **VRAM Efficiency:** This saves massive amounts of memory. You don't have to load two entire LLMs; you only load one LLM with two small "caps" on top.
* **The "Tug-of-War":** Because they share a body, the Actor wants to update the weights to be a better writer, while the Critic wants to update them to be a better judge. They are essentially "tugging" on the same set of weights.

---

###  The "Cold Start" of the Accountant
Even though TRL handles the architecture, it cannot give the Critic "knowledge" instantly.

* **The Actor:** Starts **"Smart."** It already knows how to speak because it was trained during the SFT phase.
* **The Critic:** Starts **"Blind."** Because the Value Head was randomly initialized, its first few guesses for $V(s)$ will be total nonsense.

**The Result:** During the first few batches of PPO, the `PPOTrainer` focuses heavily on the **Value Loss**. The Critic has to "sprint" to catch up to the Actor's current performance level so it can provide a stable baseline for the Advantage calculation.

---

###  Comparison: PPO vs. GRPO Architecture
This architectural overhead (and the VRAM cost of the Reference Model) is exactly what **GRPO** (Group Relative Policy Optimization) seeks to eliminate.

| Feature | PPO (via `trl`) | GRPO |
| :--- | :--- | :--- |
| **Model Type** | `AutoModelForCausalLMWithValueHead` | Standard `AutoModelForCausalLM` |
| **Extra Heads** | Yes (Scalar Value Head) | **None** |
| **Backbone Tug-of-War** | Actor vs. Critic | **None** |
| **Accountant Required?** | Yes (The Critic) | No (Group Statistics) |





In [2]:
print("Done")

Done
